In [0]:
%python
# Core EDA, identify the bronze data and what needs to be transformed for required analytics.

Derived insights:
Unique grain at: (FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID)

Tail_Number IS the aircraft registration — it uniquely identifies the physical airplane. Your intuition was correct. But here's the business reality:

Regional aircraft fly multiple legs of the same route per day




In [0]:
%sql
USE skyops.bronze

In [0]:
%python
from pyspark.sql import functions as f

In [0]:
%python
df = spark.table('skyops.bronze.bts_ontime')
df.describe().display()

In [0]:
select distinct year, month from bts_ontime

In [0]:
-- Data quality issue 1:
SELECT
 FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME, COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME HAVING Tail_Number IS NOT NULL AND dep_time is not null and COUNT(*) != 1

In [0]:
-- Main grain of the table
SELECT
 FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID, COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID HAVING FL_NUMBER IS NOT NULL AND COUNT(*) != 1

In [0]:
-- Keep in mind: there can be same plane with same origin and dest airports in the same day ofcourse! 
SELECT
 FL_DATE, AIRLINE_CODE, Tail_Number, OriginAirportID, DestAirportID, COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, Tail_Number, OriginAirportID, DestAirportID HAVING Tail_Number IS NOT NULL AND COUNT(*) != 1

In [0]:
-- keep in mind: diff flights totally can exist with same tail number and same dep time
SELECT
 *
FROM bts_ontime 
WHERE FL_DATE = 20260224 AND Tail_Number = 'N201JQ' AND CRS_DEP_TIME = 600 AND AIRLINE_CODE = 'YX'

In [0]:
-- keep in mind: multiple carriers can have the same flight number with same origin & destination in the same day
SELECT
 *
FROM bts_ontime 
WHERE FL_DATE = 20260221 AND Tail_Number = 'N495AS' AND OriginAirportID = 13830 AND DestAirportID = 12173

## Primary Key Decision: Silver/Gold Layers + Power BI

### Chosen Grain: (FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID)

**Rationale:**

1. **Zero duplicates** — unique table grain
2. **Business alignment** — matches how passengers book flights (by flight number, not tail number)
3. **Supports all 4 business questions** in SkyOps:
   * Route performance analysis (needs origin/dest)
   * Weather correlation by airport
   * Carrier delay recovery (needs flight number to track same service)
   * COVID-19 network reshaping (route-level)

4. **Handles cancelled flights** — DEP_TIME is NULL for cancellations, but FL_NUMBER always exists
5. **No data loss** — don't delete 187 "impossible" tail number records; they represent real scheduled services

### Why NOT (FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME):

* **Data quality issues**: 187 records show same aircraft departing 2 airports at same minute (physically impossible)
* **Tail_Number unreliable**: Overnight delays, maintenance swaps, BTS tracking errors
* **DEP_TIME nullable**: Cannot be primary key (NULLs for cancelled flights)
* **Wrong business grain**: Tracks physical aircraft, not scheduled passenger service
* **Requires data deletion**: Losing 187 legitimate business events

### Implementation Strategy:

**Silver Layer** (`skyops.silver.stg_bts_flights`):
```sql
-- Surrogate key for joins
CONCAT(AIRLINE_CODE, FL_NUMBER, '_', FL_DATE, '_', OriginAirportID, '_', DestAirportID) AS flight_id,

-- Data quality flag (don't delete, flag for analysis)
CASE 
  WHEN (FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME) IN (
    SELECT FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME 
    FROM source 
    GROUP BY 1,2,3,4 HAVING COUNT(*) > 1
  ) THEN 'SUSPECT_TAIL_ASSIGNMENT'
  ELSE NULL 
END AS data_quality_flag
```

**Gold Layer** (`skyops.gold.fct_flights`):
* Primary key: `flight_id` (surrogate)
* Composite unique constraint: (FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID)
* Keep `Tail_Number` as attribute for aircraft utilization analysis
* Incremental model with 35-day lookback (handles late BTS corrections)

**Power BI Model**:
* Fact table grain: `flight_id` (one row = one scheduled flight service)
* Relationships: `flight_id` → dim_date, dim_airport (origin), dim_airport (dest), dim_airline
* Tail_Number available as filter/slicer but NOT a relationship key

**Separate Aircraft Analysis** (if needed later):
* Create `fct_aircraft_utilization` with grain: (FL_DATE, Tail_Number, departure_sequence_num)
* This is a different analytical question (aircraft ops vs. passenger service performance)

## Star Schema Design: Primary/Foreign Key Example

### Fact Table: `fct_flights`

| **flight_id (PK)** | fl_date | airline_code | fl_number | origin_airport_id | dest_airport_id | date_key (FK) | origin_airport_key (FK) | dest_airport_key (FK) | airline_key (FK) | tail_number | dep_delay | arr_delay | cancelled | distance |
|-------------------|---------|--------------|-----------|-------------------|-----------------|---------------|-------------------------|----------------------|------------------|-------------|-----------|-----------|-----------|----------|
| AA1480_20260101_12892_10721 | 20260101 | AA | 1480 | 12892 | 10721 | 20260101 | 12892 | 10721 | AA | N101NN | 85 | 59 | 0 | 2611 |
| AA255_20260101_12478_12892 | 20260101 | AA | 255 | 12478 | 12892 | 20260101 | 12478 | 12892 | AA | N101NN | 71 | 86 | 0 | 2475 |
| OO4080_20260104_14262_10423 | 20260104 | OO | 4080 | 14262 | 10423 | 20260104 | 14262 | 10423 | OO | NULL | NULL | NULL | 1 | 1132 |
| OO4080_20260104_13891_10423 | 20260104 | OO | 4080 | 13891 | 10423 | 20260104 | 13891 | 10423 | OO | N307SY | 79 | 75 | 0 | 1196 |

**Notes:**
* `flight_id` = **Primary Key** (surrogate, human-readable)
* Composite uniqueness enforced: (fl_date, airline_code, fl_number, origin_airport_id, dest_airport_id)
* Same flight number (OO4080) on same day, different origins → distinct records ✓
* `tail_number` is an **attribute** (not a key) — can be NULL for cancelled flights

---

### Dimension Table: `dim_date`

| **date_key (PK)** | full_date | year | month | day | day_of_week | is_weekend | is_holiday | month_name |
|-------------------|-----------|------|-------|-----|-------------|------------|------------|------------|
| **20260101** | 2026-01-01 | 2026 | 1 | 1 | Thursday | 0 | 1 | January |
| **20260104** | 2026-01-04 | 2026 | 1 | 4 | Sunday | 1 | 0 | January |
| **20260105** | 2026-01-05 | 2026 | 1 | 5 | Monday | 0 | 0 | January |

**Relationship:** `fct_flights.date_key` → `dim_date.date_key` (many-to-one)

---

### Dimension Table: `dim_airport`

| **airport_key (PK)** | iata_code | icao_code | airport_name | city | state | timezone | hub_class | latitude | longitude |
|----------------------|-----------|-----------|--------------|------|-------|----------|-----------|----------|----------|
| **10721** | BOS | KBOS | Boston Logan International | Boston | MA | America/New_York | Large Hub | 42.3656 | -71.0096 |
| **10423** | AUS | KAUS | Austin-Bergstrom International | Austin | TX | America/Chicago | Medium Hub | 30.1945 | -97.6699 |
| **12892** | LAX | KLAX | Los Angeles International | Los Angeles | CA | America/Los_Angeles | Large Hub | 33.9425 | -118.4081 |
| **12478** | JFK | KJFK | John F Kennedy International | New York | NY | America/New_York | Large Hub | 40.6413 | -73.7781 |
| **13891** | ONT | KONT | Ontario International | Ontario | CA | America/Los_Angeles | Medium Hub | 34.0560 | -117.6012 |
| **14262** | PSP | KPSP | Palm Springs International | Palm Springs | CA | America/Los_Angeles | Small Hub | 33.8297 | -116.5067 |

**Relationships:**
* `fct_flights.origin_airport_key` → `dim_airport.airport_key` (many-to-one)
* `fct_flights.dest_airport_key` → `dim_airport.airport_key` (many-to-one)

**Note:** Same airport can be origin for some flights, destination for others (role-playing dimension)

---

### Dimension Table: `dim_airline`

| **airline_key (PK)** | airline_name | dot_code | is_low_cost_carrier | alliance | headquarters |
|----------------------|--------------|----------|---------------------|----------|-------------|
| **AA** | American Airlines | 19805 | 0 | Oneworld | Fort Worth, TX |
| **OO** | SkyWest Airlines | 20304 | 0 | NULL | St. George, UT |
| **DL** | Delta Air Lines | 19790 | 0 | SkyTeam | Atlanta, GA |
| **WN** | Southwest Airlines | 19393 | 1 | NULL | Dallas, TX |

**Relationship:** `fct_flights.airline_key` → `dim_airline.airline_key` (many-to-one)

---

## Power BI Relationships

```
      dim_date                dim_airline
     [date_key] ──────┐      [airline_key] ────┐
                       │                        │
                       ├──► fct_flights ◄───────┤
                       │     [flight_id]        │
     [airport_key] ────┤     [date_key]         │
    dim_airport        │     [origin_airport_key]
                       │     [dest_airport_key] │
                       │     [airline_key]      │
                       └─────────────────────────┘
```

**Cardinalities:**
* dim_date → fct_flights: **1:Many** (one date, many flights)
* dim_airport → fct_flights (origin): **1:Many** (one origin airport, many departures)
* dim_airport → fct_flights (dest): **1:Many** (one destination airport, many arrivals)
* dim_airline → fct_flights: **1:Many** (one carrier, many flights)

**Cross-filter direction:** Single (dimension → fact)

---

## SQL Implementation Example

```sql
-- Silver staging table
CREATE OR REPLACE TABLE skyops.silver.stg_bts_flights AS
SELECT
  -- Surrogate PK
  CONCAT(AIRLINE_CODE, FL_NUMBER, '_', FL_DATE, '_', OriginAirportID, '_', DestAirportID) AS flight_id,
  
  -- Natural composite key (for uniqueness constraint)
  FL_DATE,
  AIRLINE_CODE,
  FL_NUMBER,
  OriginAirportID,
  DestAirportID,
  
  -- Foreign keys
  CAST(FL_DATE AS STRING) AS date_key,
  OriginAirportID AS origin_airport_key,
  DestAirportID AS dest_airport_key,
  AIRLINE_CODE AS airline_key,
  
  -- Attributes
  Tail_Number,
  DEP_DELAY,
  ARR_DELAY,
  CANCELLED,
  DISTANCE,
  
  -- Data quality flag
  CASE 
    WHEN (FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME) IN (
      SELECT FL_DATE, AIRLINE_CODE, Tail_Number, DEP_TIME 
      FROM skyops.bronze.bts_ontime 
      WHERE Tail_Number IS NOT NULL AND DEP_TIME IS NOT NULL
      GROUP BY 1,2,3,4 HAVING COUNT(*) > 1
    ) THEN 'SUSPECT_TAIL_ASSIGNMENT'
    ELSE NULL 
  END AS data_quality_flag
  
FROM skyops.bronze.bts_ontime;

-- Add unique constraint
ALTER TABLE skyops.silver.stg_bts_flights
ADD CONSTRAINT unique_flight_service 
UNIQUE (FL_DATE, AIRLINE_CODE, FL_NUMBER, OriginAirportID, DestAirportID);
```

In [0]:
-- Demonstrate the schema with sample data

-- 1. Create sample dim_date
CREATE OR REPLACE TEMPORARY VIEW sample_dim_date AS
SELECT
  20260101 AS date_key,
  DATE '2026-01-01' AS full_date,
  2026 AS year,
  1 AS month,
  1 AS day,
  'Thursday' AS day_of_week,
  0 AS is_weekend
UNION ALL
SELECT 20260104, DATE '2026-01-04', 2026, 1, 4, 'Sunday', 1;

-- 2. Create sample dim_airport
CREATE OR REPLACE TEMPORARY VIEW sample_dim_airport AS
SELECT 10721 AS airport_key, 'BOS' AS iata_code, 'Boston Logan Intl' AS airport_name, 'Boston' AS city, 'MA' AS state, 'Large Hub' AS hub_class
UNION ALL SELECT 10423, 'AUS', 'Austin-Bergstrom Intl', 'Austin', 'TX', 'Medium Hub'
UNION ALL SELECT 12892, 'LAX', 'Los Angeles Intl', 'Los Angeles', 'CA', 'Large Hub'
UNION ALL SELECT 12478, 'JFK', 'John F Kennedy Intl', 'New York', 'NY', 'Large Hub'
UNION ALL SELECT 13891, 'ONT', 'Ontario Intl', 'Ontario', 'CA', 'Medium Hub'
UNION ALL SELECT 14262, 'PSP', 'Palm Springs Intl', 'Palm Springs', 'CA', 'Small Hub';

-- 3. Create sample dim_airline
CREATE OR REPLACE TEMPORARY VIEW sample_dim_airline AS
SELECT 'AA' AS airline_key, 'American Airlines' AS airline_name, 19805 AS dot_code, 0 AS is_low_cost_carrier, 'Oneworld' AS alliance
UNION ALL SELECT 'OO', 'SkyWest Airlines', 20304, 0, NULL;

-- 4. Create sample fact table with relationships
CREATE OR REPLACE TEMPORARY VIEW sample_fct_flights AS
SELECT
  'AA1480_20260101_12892_10721' AS flight_id,
  '20260101' AS fl_date,
  'AA' AS airline_code,
  1480 AS fl_number,
  12892 AS origin_airport_id,
  10721 AS dest_airport_id,
  20260101 AS date_key,
  12892 AS origin_airport_key,
  10721 AS dest_airport_key,
  'AA' AS airline_key,
  'N101NN' AS tail_number,
  85 AS dep_delay,
  59 AS arr_delay,
  2611 AS distance
UNION ALL
SELECT 'OO4080_20260104_14262_10423', '20260104', 'OO', 4080, 14262, 10423, 20260104, 14262, 10423, 'OO', NULL, NULL, NULL, 1132
UNION ALL
SELECT 'OO4080_20260104_13891_10423', '20260104', 'OO', 4080, 13891, 10423, 20260104, 13891, 10423, 'OO', 'N307SY', 79, 75, 1196;

-- 5. Demonstrate star schema query
SELECT
  f.flight_id,
  d.full_date,
  d.day_of_week,
  al.airline_name,
  orig.iata_code AS origin,
  orig.city AS origin_city,
  dest.iata_code AS destination,
  dest.city AS dest_city,
  f.tail_number,
  f.dep_delay,
  f.arr_delay,
  f.distance,
  orig.hub_class AS origin_hub_class
FROM sample_fct_flights f
  INNER JOIN sample_dim_date d ON f.date_key = d.date_key
  INNER JOIN sample_dim_airline al ON f.airline_key = al.airline_key
  INNER JOIN sample_dim_airport orig ON f.origin_airport_key = orig.airport_key
  INNER JOIN sample_dim_airport dest ON f.dest_airport_key = dest.airport_key
ORDER BY f.flight_id